# Day 12 · Power — Who Fixes the Sidewalk

**Question.** CTA decides the elevator. The last block, from the front door to the bus stop or the
station, is sidewalk and curb, and in Chicago each of the 50 aldermen chooses how to spend the ward's
yearly menu money on it. Who uses which kind of power to get a broken sidewalk fixed?

**Approach.** Three ways a sidewalk gets fixed, per ward, 2019–2022 (four full years on one ward map):
1. **The alderman:** menu program spending by category (Ward Wise's compilation of the City's menu
   reports), share on sidewalks and crossings.
2. **The complaint:** 311 sidewalk inspection requests filed, still open, and how long they took.
3. **The wallet:** 311 applications to the Shared Cost Sidewalk Program, where the owner pays part.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Ward Wise menu data (pinned commit), precincts 2013–2022, 311 requests, 2020 blocks |
| 3. Validate | Counts, years, ward coverage |
| 4. Transform | Wards from precincts; points and projects to wards; address units |
| 5. Analyze | Ward table, classes, correlations, the finding |
| 6. QA | Checks and a quick look |
| 7. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
from datetime import date
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
import shapely
from shapely.geometry import mapping, shape

from atlas import grid, socrata

logging.basicConfig(level=logging.INFO, format='%(message)s')
DAY = Path('.')
ROOT = DAY.resolve().parents[1]
RAW = DAY / 'data' / 'raw'
OUT = DAY / 'output'
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)

YEARS = (2019, 2022)          # one ward map (May 2015 to May 2023), four full years
SIMPLIFY = 20                 # address units (1/40 mile) for ward outlines
WW_SHA = '021c865eed680a45c556bc4435e0d1f2af228e7e'   # ward-wise/data-analysis, 2024-09-02
WW = f'https://raw.githubusercontent.com/ward-wise/data-analysis/{WW_SHA}/data/'
GROUP = {'Sidewalk Repair': 'sidewalks', 'Pedestrian Infrastructure': 'crossings',
         'Street Resurfacing': 'streets', 'Alleys': 'streets', 'Street Redesign': 'streets', 'Viaducts': 'streets',
         'Lighting': 'lights', 'Traffic Signals': 'lights', 'Police Cameras': 'cameras'}
CATS = ['sidewalks', 'crossings', 'streets', 'lights', 'cameras', 'other']
MENU_BREAKS = [0.02, 0.05, 0.07, 0.10, 0.15, 0.20]   # share on sidewalks + crossings, seven classes

## 2. Pull

In [ ]:
def pull(url: str, path: Path) -> Path:
    if not path.exists():
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path

menu_csv = pull(WW + '2005-2023%20menu%20spending.csv', RAW / 'wardwise_menu_2005-2023.csv')
menu_geo = pull(WW + '2019-2022%20data_geocoded.geojson', RAW / 'wardwise_menu_2019-2022_geocoded.geojson')
prec = pull('https://data.cityofchicago.org/resource/nvke-umup.geojson?$limit=5000', RAW / 'ward_precincts_2013-2022.geojson')

SR_TYPES = ('Sidewalk Inspection Request', 'Shared Cost Sidewalk Program Request')
sr_files = sorted(RAW.glob('sr311_sidewalk_*.csv.gz'))
if sr_files:
    sr_path = sr_files[-1]
else:
    where = ' OR '.join(f"sr_type='{t}'" for t in SR_TYPES)
    df = socrata.fetch('v6vf-nfxy', select='sr_number,sr_type,status,created_date,closed_date,latitude,longitude', where=where)
    sr_path = RAW / f'sr311_sidewalk_{date.today().isoformat()}.csv.gz'
    df.to_csv(sr_path, index=False)
pulled311 = sr_path.name.split('_')[-1].removesuffix('.csv.gz')
sr = pd.read_csv(sr_path)
menu = pd.read_csv(menu_csv)
blocks = pd.read_csv(ROOT / 'reference' / 'chicago-blocks-2020.csv')
print(menu.shape, sr.shape, blocks.shape, pulled311)

## 3. Validate

In [ ]:
menu['ward'] = menu.ward.astype(float).astype(int)
print('rows outside wards 1-50 (2023 ward 99, outside the study years):', int((~menu.ward.between(1, 50)).sum()))
m = menu[menu.year.between(*YEARS)].copy()
assert set(m.ward) == set(range(1, 51))
m['grp'] = m.category.map(GROUP).fillna('other')
assert m.cost.notna().all() and (m.cost >= 0).all()
print(m.groupby('year').cost.agg(['count', 'sum']).round(0))
print(m.category.value_counts())
print(sr.groupby(['sr_type', 'status']).size())
print('311 without coordinates:', sr.latitude.isna().sum())

## 4. Transform

In [ ]:
pr = gpd.read_file(prec)
pr['ward'] = pr.ward.astype(int)
wards = pr.dissolve('ward')[['geometry']].reset_index()
wards['geometry'] = wards.geometry.buffer(0)
assert len(wards) == 50

def join(df, lon, lat):
    pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon].astype(float), df[lat].astype(float)), crs=4326)
    right = wards[['ward', 'geometry']].rename(columns={'ward': 'w'}).set_crs(4326, allow_override=True)
    j = gpd.sjoin(pts.drop(columns=['ward'], errors='ignore'), right, predicate='within', how='left')
    return pd.DataFrame(j.drop(columns=['geometry', 'index_right'], errors='ignore'))

bl = join(blocks, 'lon', 'lat')
print('blocks outside a ward:', bl.w.isna().sum(), 'residents', bl.loc[bl.w.isna(), 'pop'].sum())

sr = sr.dropna(subset=['latitude', 'longitude'])
sr['created'] = pd.to_datetime(sr.created_date)
sr['closed'] = pd.to_datetime(sr.closed_date)
sr = join(sr, 'longitude', 'latitude')
print('311 outside a ward:', sr.w.isna().sum())
sr = sr.dropna(subset=['w'])
sr['w'] = sr.w.astype(int)
now = pd.Timestamp(pulled311)
insp = sr[sr.sr_type == SR_TYPES[0]].copy()
shared = sr[sr.sr_type == SR_TYPES[1]].copy()
insp['age_y'] = (now - insp.created).dt.days / 365.25
in_years = lambda d: d[d.created.dt.year.between(*YEARS)]

# address units and grid miles
def to_addr(geom):
    return shapely.transform(geom, lambda xy: np.c_[grid.to_address(xy[:, 1], xy[:, 0])])

def addr_to_miles(y):
    y = np.asarray(y, float)
    return np.where(y >= 0, y / 800, np.where(y >= -1200, y / 1200, -1 + (y + 1200) / 800))

wards['addr'] = [to_addr(g).simplify(SIMPLIFY) for g in wards.geometry]
rep = [to_addr(g.representative_point()) for g in wards.geometry]
wards['px'] = [p.x for p in rep]
wards['py'] = [p.y for p in rep]

# ward-number boxes: the spot where a box the size of the plate's label (the tightest case: 25 px a
# mile there) sits farthest inside its ward, measured in grid miles (the drawing space). Wards do
# not overlap, so boxes kept inside their own ward never touch another box or another ward.
PX_PER_MILE = 25.0
BOX_H = 9.0 / PX_PER_MILE                       # 9 px tall
box_w = lambda n: (4.0 + 3.7 * len(str(n))) / PX_PER_MILE   # Roboto 600 at 6.5 px, 2 px padding
MARGIN = 1.2 / PX_PER_MILE                      # keep 1.2 px of ward around the box

def to_miles_geom(g):
    return shapely.transform(g, lambda xy: np.c_[xy[:, 0] / 800, addr_to_miles(xy[:, 1])])

def label_spot(ward, geom_addr, scale=1.0, margin=MARGIN):
    poly = to_miles_geom(geom_addr)
    w, h = box_w(ward) * scale, BOX_H * scale
    inner = poly.buffer(-margin)
    best, best_d = None, -1.0
    parts = sorted(getattr(inner, 'geoms', [inner]), key=lambda g: -g.area)
    for part in parts[:3]:
        if part.is_empty:
            continue
        x0, y0, x1, y1 = part.bounds
        step = 0.04
        for x in np.arange(x0 + w / 2, x1 - w / 2 + 1e-9, step):
            for y in np.arange(y0 + h / 2, y1 - h / 2 + 1e-9, step):
                box = shapely.box(x - w / 2, y - h / 2, x + w / 2, y + h / 2)
                if not part.contains(box):
                    continue
                d = part.exterior.distance(box) if part.geom_type == 'Polygon' else part.boundary.distance(box)
                if d > best_d:
                    best, best_d = (x, y), d
    if best is None:                            # no room for the whole box: fall back to the polylabel
        pl = shapely.ops.polylabel(max(getattr(poly, 'geoms', [poly]), key=lambda g: g.area), tolerance=0.01)
        best, best_d = (pl.x, pl.y), -1.0
    return best, best_d

import shapely.ops
spots, lscale = {}, {}
for r in wards.itertuples():
    for sc, mg in ((1.0, MARGIN), (0.85, MARGIN), (0.75, 0.6 / PX_PER_MILE), (0.65, 0.4 / PX_PER_MILE)):
        spot = label_spot(int(r.ward), r.addr, sc, mg)
        if spot[1] >= 0:
            break
    spots[int(r.ward)], lscale[int(r.ward)] = spot, sc
print('smaller boxes:', {w: sc for w, sc in lscale.items() if sc < 1})
wards['lx'] = [spots[int(w)][0][0] * 800 for w in wards.ward]                       # back to address units
wards['ly'] = [float(grid.north_to_address(spots[int(w)][0][1])) for w in wards.ward]
tight = {w: round(d * PX_PER_MILE, 1) for w, (_, d) in spots.items() if d < 0.5 / PX_PER_MILE}
print('label clearance (plate px), smallest:', sorted(((round(d * PX_PER_MILE, 1), w) for w, (_, d) in spots.items()))[:6])
print('wards without room for the full box:', [w for w, (_, d) in spots.items() if d < 0])

# geocoded sidewalk and crossing projects, for drawing
feats = json.load(open(menu_geo))['features']
lines, ticks = [], []
n_rows = 0
for f in feats:
    p = f['properties']
    if GROUP.get(p['category']) not in ('sidewalks', 'crossings'):
        continue
    n_rows += 1
    if not f['geometry']:
        continue
    w = int(float(p['ward']))
    g = to_addr(shape(f['geometry']))
    parts = getattr(g, 'geoms', [g])
    for part in parts:
        if part.geom_type == 'LineString':
            lines.append([w, [[round(x), round(y)] for x, y in part.coords]])
        elif part.geom_type == 'Point':
            ticks.append([w, round(part.x), round(part.y)])
        elif part.geom_type == 'Polygon':
            lines.append([w, [[round(x), round(y)] for x, y in part.exterior.coords]])
        elif part.geom_type in ('MultiLineString', 'MultiPoint', 'MultiPolygon', 'GeometryCollection'):
            for q in part.geoms:
                if q.geom_type == 'Point':
                    ticks.append([w, round(q.x), round(q.y)])
                elif q.geom_type == 'LineString':
                    lines.append([w, [[round(x), round(y)] for x, y in q.coords]])
print('sidewalk/crossing rows in geojson', n_rows, '| lines', len(lines), '| points', len(ticks))

## 5. Analyze

In [ ]:
spend = m.pivot_table(index='ward', columns='grp', values='cost', aggfunc='sum', fill_value=0).reindex(columns=CATS, fill_value=0)
spend['total'] = spend.sum(axis=1)
people = bl.groupby('w').agg(pop=('pop', 'sum'), amb=('amb', 'sum'))
fi = in_years(insp)
req = pd.DataFrame({
    'filed': fi.groupby('w').size(),
    'open': insp[insp.status == 'Open'].groupby('w').size(),
    'open3': insp[(insp.status == 'Open') & (insp.age_y >= 3)].groupby('w').size(),
    'closeDays': fi.assign(d=(fi.closed - fi.created).dt.days).groupby('w').d.median(),
    'shared': in_years(shared).groupby('w').size(),
}).fillna(0)
T = spend.join(people).join(req).fillna(0)
T.index = T.index.astype(int)
T['walk'] = T.sidewalks + T.crossings
T['share'] = T.walk / T.total
T['filed1k'] = T.filed / T['pop'] * 1000
T['shared1k'] = T.shared / T['pop'] * 1000
T['ambShare'] = T.amb / T['pop']
T['cls'] = np.searchsorted(MENU_BREAKS, T.share, side='right')
# the scatter inset: thirds of the wards on each axis (the bivariate classes in colour mode)
qv = np.quantile(T.filed1k, [1 / 3, 2 / 3]); qm = np.quantile(T.shared1k, [1 / 3, 2 / 3])
T['tVoice'] = np.searchsorted(qv, T.filed1k, side='right')
T['tMoney'] = np.searchsorted(qm, T.shared1k, side='right')
print(pd.crosstab(T.tMoney, T.tVoice))

sp = lambda a, b: round(float(T[a].corr(T[b], method='spearman')), 2)
corr = dict(share_amb=sp('share', 'ambShare'), share_filed=sp('share', 'filed1k'), filed_shared=sp('filed1k', 'shared1k'),
            share_shared=sp('share', 'shared1k'))
low = T[T.share < 0.01]                            # the wards that spent under 1% on sidewalks and crossings
lowRank = dict(filed=[int((T.filed1k > r.filed1k).sum()) + 1 for r in low.itertuples()],
               shared=[int((T.shared1k > r.shared1k).sum()) + 1 for r in low.itertuples()])
top, bot = T.share.idxmax(), T.share.idxmin()
per_year = m.groupby(['ward', 'year']).cost.sum().unstack()
alloc = {int(y): float(per_year[y].median()) for y in per_year}
summary = dict(
    alloc=alloc, wards=50, residents=int(T['pop'].sum()), years=list(YEARS),
    menu=float(T.total.sum()), walk=float(T.walk.sum()), walkShare=float(T.walk.sum() / T.total.sum()),
    byCat={c: float(T[c].sum()) for c in CATS},
    perWardMedian=float(T.total.median()),
    top=dict(ward=int(top), share=float(T.share[top])), bottom=dict(ward=int(bot), share=float(T.share[bot])),
    zeroWards=[int(w) for w in T.index[T.share < 0.01]],
    filed=int(T.filed.sum()), open=int(T.open.sum()), open3=int(T.open3.sum()), shared=int(T.shared.sum()),
    closeDays=[float(T.closeDays.min()), float(T.closeDays.median()), float(T.closeDays.max())],
    closeWards=[int(T.closeDays.idxmin()), int(T.closeDays.idxmax())],
    low=[int(w) for w in low.index], lowFiled1k=[float(low.filed1k.min()), float(low.filed1k.max())],
    lowShared=int(low.shared.sum()), lowSharedShare=float(low.shared.sum() / T.shared.sum()),
    lowRank=lowRank, corr=corr,
    projectsDrawn=len(lines) + len(ticks),
)
pct = lambda v: f'{round(v * 100)}%'
lw = summary['low']
words = {2: 'two', 3: 'three', 4: 'four', 5: 'five'}
n_low = words.get(len(low), str(len(low)))
finding = (f'Every alderman gets the same menu money, about ${T.total.median() / 1e6:.1f} million from {YEARS[0]} to {YEARS[1]}. '
           f'Ward {top} put {pct(T.share[top])} of it into sidewalks and crossings; Ward {bot} put {"none" if T.share[bot] < 0.005 else pct(T.share[bot])}. '
           f'In the {n_low} wards that put in less than 1%, residents filed among the most sidewalk complaints and led the city in applications to share the cost of a repair.')
print(finding)
print(json.dumps(summary, indent=1)[:1500])
print(T[['share', 'filed1k', 'shared1k', 'open', 'open3', 'closeDays', 'ambShare']].sort_values('share').round(3).head(8))

## 6. QA

In [ ]:
# menu totals per ward and year should sit near the yearly allocation; flag outliers
per_year = m.groupby(['ward', 'year']).cost.sum().unstack()
print(per_year.describe().round(0))
# the 311 ward field (current map) vs our 2015-map assignment, for requests filed 2019-2022
print('requests in the lowest-share wards (open, 3+ years):', T.loc[lw, ['open', 'open3']].to_dict())
# every sidewalk/crossing project row vs geometry drawn
print('menu sidewalk + crossing rows 2019-22:', int(m.grp.isin(['sidewalks', 'crossings']).sum()), '| drawn:', summary['projectsDrawn'])

## 7. Export

In [ ]:
def rings(g):
    polys = getattr(g, 'geoms', [g])
    return [[[round(x), round(y)] for x, y in r.coords] for p in polys for r in [p.exterior, *p.interiors]]

rows = []
for r in wards.itertuples():
    t = T.loc[r.ward]
    rows.append(dict(
        ward=int(r.ward), rings=rings(r.addr), cx=round(r.px), cy=round(r.py), lx=round(r.lx), ly=round(r.ly), ls=lscale[int(r.ward)],
        pop=int(t['pop']), amb=round(float(t.amb)), total=round(float(t.total)),
        **{c: round(float(t[c])) for c in CATS}, share=round(float(t.share), 4), cls=int(t.cls),
        filed=int(t.filed), open=int(t.open), open3=int(t.open3), closeDays=float(t.closeDays), shared=int(t.shared),
        filed1k=round(float(t.filed1k), 2), shared1k=round(float(t.shared1k), 2), tVoice=int(t.tVoice), tMoney=int(t.tMoney),
    ))
plate = dict(
    finding=finding, summary=summary, wards=rows,  # project geometry no longer drawn (user, 2026-11-14)
    breaks=dict(menu=MENU_BREAKS, voice=[round(v, 2) for v in qv], money=[round(v, 2) for v in qm]),
    pulled311=pulled311,
    sources=dict(wardwise=f'https://github.com/ward-wise/data-analysis/tree/{WW_SHA}/data',
                 obm='https://www.chicago.gov/city/en/depts/obm/provdrs/cap_improve/svcs/cip-archive.html',
                 sr311='https://data.cityofchicago.org/d/v6vf-nfxy', precincts='https://data.cityofchicago.org/d/nvke-umup'),
)
(OUT / 'plate.json').write_text(json.dumps(plate, separators=(',', ':')))
print('plate.json', round((OUT / 'plate.json').stat().st_size / 1024), 'KB')